# 1. 들어가며
오늘은 실전 프로젝트를 가정하고 Huggingface transformers framework를 활용하여 빠르게 자신만의 커스텀 프로젝트를 구성해 보는 실습을 진행해 보겠습니다. framework 내의 Model, Tokenizer, Trainer 등이 어떻게 활용되는지 꼼꼼히 살펴봅시다. 하나의 framework에 익숙해진다면 다른 framework에 적응하는 것도 훨씬 수월해질 것입니다. 또한 NLP 분야의 best practice까지 자연스럽게 체득하면서 어느새 연구개발 실력이 불쑥 자라있는 자신을 발견하게 될 것입니다.

## 학습 목표
- GLUE benchmark에 포함된 여러가지 task를 파악하고, 상황에 맞는 것을 취사선택할 수 있다.
- 필요한 Custom dataset을 직접 만들고, 이를 훈련시킬 수 있다.
- 더 나아가, Huggingface framework 사용법을 익히고, 이를 편하게 이용할 수 있다.

## 오늘의 목차
- 1. GLUE dataset과 Huggingface
    - NLP를 대표하는 GLUE task에 무엇이 있는지 배워봅시다.
- 2. 커스텀 프로젝트 제작
    - (1) Dataset
        - 구슬이 서 말이라도 꿰어야 보배다! 데이터를 task에 맞게 가공해봅시다.
    - (2) Tokenizer와 Model
        - Custom project를 위한 모델과 tokenizer을 불러와봅시다.
    - (3) Train/Evaluation과 Test
        - Keras와 Huggingface 두 가지 방식으로 훈련해보아요.

# 2. GLUE dataset과 Huggingface
## GLUE Benchmark Dataset
---
Pretrained model의 성능을 측정하기 위해 최근은 SQuAD 등 기존에 유명한 데이터셋 한 가지만 가지고 성능을 논하는 것이 아니라, classification, summarization, reasoning, Q&A 등 NLP 모델의 성능을 평가할 수 있는 다양한 task를 해당 모델 하나만을 이용해 모두 수행해 보면서 종합적인 성능을 논하는 것이 일반화되었습니다.

그중 NLP 모델의 성능을 측정하기 위한 데이터셋으로 최근 활용되는 대표적인 것 중에 [General Language Understanding Evaluation(GLUE) benchmark Dataset](https://gluebenchmark.com/)이 있습니다. GLUE에는 총 11가지 데이터셋이 있습니다. 각각의 개요는 다음과 같습니다.

- CoLA : 문법에 맞는 문장인지 판단
- MNLI : 두 문장의 관계 판단(entailment, contradiction, neutral)
- MNLI-MM : 두 문장이 안 맞는지 판단
- MRPC : 두 문장의 유사도 평가
- SST-2 : 감정분석
- STS-B : 두 문장의 유사도 평가
- QQP : 두 질문의 유사도 평가
- QNLI : 질문과 paragraph 내 한 문장이 함의 관계(entailment)인지 판단
- RTE : 두 문장의 관계 판단(entailment, not_entailment)
- WNLI : 원문장과 대명사로 치환한 문장 사이의 함의 관계 판단
- Diagnostic Main : 자연어 추론 문제를 통한 문장 이해도 평가

GLUE 홈페이지에는 위 [11가지 task에 대한 상세한 설명](https://gluebenchmark.com/tasks), 그리고 [Leaderboard](https://gluebenchmark.com/leaderboard)를 운영하고 있습니다. 한 가지 task에만 최적화된 모델이 아니라, 다양한 형태의 문제를 골고루 잘 푸는 모델을 찾기 위한 노력이 계속되고 있습니다.

주관식 퀴즈

Q. GLUE benchmark에서 각 태스크에 사용된 metric으로는 어떠한 것들이 있었나요?
답변을 적어보세요 😀

힌트

Accuracy, F1-score, Pearson-Spearman Correlation, MCC(Matthew's Correlation Coefficient) 등이 사용되었습니다.

## Huggingface transformers 설치 및 환경 구성
---
터미널에 아래 명령어를 입력하여 환경이 잘 구성됐는지 확인해주세요.

1~2분이 소요될 수 있습니다.
```
$ pip install transformers
$ python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('I love you'))"
```

혹시 위의 코드가 잘 실행되지 않을 경우 터미널을 열고 아래 명령어를 입력하여 환경을 구성합니다.
```
$ pip uninstall transformers -y
$ pip install transformers
$ mkdir -p transformers
```

잘 실행되시나요? 위 코드는 11가지 GLUE task 중 감정분석을 수행하는 예제 코드입니다. 명령어 실행 결과를 보니 ‘I love you’ 문장이 99%의 확률로 positive 라고 판단했군요.

---
이번 노드에서는 GLUE의 'mrpc' task를 나만의 커스텀 프로젝트로 구성해서 해결해볼 예정입니다. 이 과정을 통해 Huggingface framework에 대해 좀 더 명확하게 이해하실 수 있을 것입니다.

In [1]:
!pip install transformers
!pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [transformers] [transformers]ub]


In [2]:
!python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('I love you'))"

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
config.json: 100%|█████████████████████████████| 629/629 [00:00<00:00, 3.07MB/s]
model.safetensors: 100%|█████████████████████| 268M/268M [00:04<00:00, 64.5MB/s]
Loading weights: 100%|██████████████████████| 104/104 [00:00<00:00, 3964.87it/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 273kB/s]
vocab.txt: 232kB [00:00, 41.6MB/s]
[{'label': 'POSITIVE', 'score': 0.9998656511306763}]


# 3. 커스텀 프로젝트 제작
## (1) Datasets
### Huggingface dataset에서 불러오기
---
Huggingface dataset에는 상당한 양의 데이터가 있습니다. datasets을 이용하면 손쉽게 데이터를 불러올 수 있습니다.

In [3]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 102.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 20.0.0
    Uninstalling pyarrow-20.0.0:
      Successfully uninstalled pyarrow-20.0.0━━━━━━━━━━━━━━━━━━━━━  1/11 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [datasets]/11 [datasets]]ss]


- huggingface mrpc dataset : Hugging Face의 MRPC 데이터셋은 Microsoft Research Paraphrase Corpus의 약자로, GLUE 벤치마크의 일부인 문장 쌍 분류 데이터셋입니다. 이 데이터셋은 약 5,800개의 영어 문장 쌍으로 구성되어 있으며, 각 쌍이 의미적으로 동일한 paraphrase인지 여부를 이진 분류(0 또는 1 라벨)하는 태스크를 제공합니다.

In [4]:
import datasets
from datasets import load_dataset

huggingface_mrpc_dataset = load_dataset('glue', 'mrpc')
print(huggingface_mrpc_dataset)

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


huggingface mrpc dataset을 확인해보면 위와 같이 구성되어 있습니다.

Dataset dictionary안에 train dataset, validation dataset, test dataset으로 구성되어 있고 각 Dataset은 ‘sentence1’, ‘sentence2’, ‘label’, ‘idx’(인덱스)로 구성되어 있습니다. 해당 내용처럼 Huggingface datasets를 사용하면 손쉽게 모델의 input으로 사용할 수 있다는 장점이 있습니다.

주관식 퀴즈

Q. MRPC는 GLUE benchmark 중 어떤 task를 수행하는 데이터셋인지 기억나시나요? 떠오른 내용을 한 번 적어봅시다!

힌트

MRPC(The Microsoft Research Paraphrase Corpus)는 주어진 두 문장이 동일한 의미를 가지고 있는지 여부를 분류하는 작업에 사용됩니다. Label은 0(not equivalent, 상이함)과 1(equivalent, 동일함) 두 가지로 나뉩니다.

Train datasets의 각 컬럼에 해당하는 요소들을 몇 가지만 뜯어보며, 이것들이 제대로 짝을 지어있는지 확인해볼까요?

In [5]:
train = huggingface_mrpc_dataset['train']
cols = train.column_names
cols

['sentence1', 'sentence2', 'label', 'idx']

In [6]:
for i in range(5):
    for col in cols:
        print(col, ":", train[col][i])
    print('\n')

sentence1 : Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
sentence2 : Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
label : 1
idx : 0


sentence1 : Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
sentence2 : Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .
label : 0
idx : 1


sentence1 : They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
sentence2 : On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .
label : 1
idx : 2


sentence1 : Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
sentence2 : Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
label : 0

1. Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
2. Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .

두 문장의 순서만 도치되었을 뿐, 의미가 같기에 label : 1로 동일한 문장이라 평가할 수 있겠네요!

### 커스텀 데이터셋 만들기
---
Huggingface에 원하는 데이터셋이 없다면 어떡하죠? 걱정마세요, Huggingface datasets API를 활용하면 데이터셋을 직접 만들 수도 있고, 업로드까지도 가능합니다!

이번에는 MRPC 데이터셋을 직접 만들어보도록 하겠습니다. GLUE 데이터셋은 [홈페이지](https://www.microsoft.com/en-us/download/details.aspx?id=52398)에서도 원본을 다운로드할 수 있습니다. 이미 학습환경에 준비된 파일로  Huggingface dataset에 맞게 가공해보겠습니다.



In [7]:
import pandas as pd
from datasets import Dataset

def parse_mrpc_file(file_path):
    """MRPC 파일을 안전하게 파싱하는 함수"""
    data = {
        'Quality': [],
        '#1 ID': [],
        '#2 ID': [], 
        '#1 String': [],
        '#2 String': []
    }
    
    with open(file_path, 'r', encoding='utf-8') as f:
        # 헤더 스킵
        next(f)
        
        for line_num, line in enumerate(f, 1):
            try:
                parts = line.strip().split('\t')
                if len(parts) >= 5:
                    # 5개 컬럼으로 분할 (마지막 탭들은 모두 마지막 컬럼에 포함)
                    quality = int(parts[0])
                    id1 = int(parts[1]) 
                    id2 = int(parts[2])
                    string1 = parts[3]
                    string2 = '\t'.join(parts[4:])  # 나머지 모든 부분을 합침
                    
                    data['Quality'].append(quality)
                    data['#1 ID'].append(id1)
                    data['#2 ID'].append(id2)
                    data['#1 String'].append(string1)
                    data['#2 String'].append(string2)
                else:
                    print(f"Line {line_num}: Invalid format, skipping")
            except Exception as e:
                print(f"Line {line_num}: Error {e}, skipping")
    
    return pd.DataFrame(data)

# 로컬 파일에서 MRPC 데이터 읽기
import os

data_dir = os.path.expanduser('~/work/unsupervised/data')
file1 = os.path.join(data_dir, 'msr_paraphrase_train.txt')
file2 = os.path.join(data_dir, 'msr_paraphrase_test.txt')

train_df = parse_mrpc_file(file1)
test_df = parse_mrpc_file(file2)

tfds의 MRPC 데이터셋은 앞서 살펴본 Huggingface dataset과는 어떤 차이가 있는지 확인해보겠습니다.

In [8]:
# 데이터 구조 확인
print("Train dataset columns:", train_df.columns.tolist())
print("Train dataset shape:", train_df.shape)
print("\nFirst 5 examples:")

for i in range(5):
    row = train_df.iloc[i]
    for col in train_df.columns:
        print(f"{col}: {row[col]}")
    print('\n')

Train dataset columns: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String']
Train dataset shape: (4076, 5)

First 5 examples:
Quality: 1
#1 ID: 702876
#2 ID: 702977
#1 String: Amrozi accused his brother, whom he called "the witness", of deliberately distorting his evidence.
#2 String: Referring to him as only "the witness", Amrozi accused his brother of deliberately distorting his evidence.


Quality: 0
#1 ID: 2108705
#2 ID: 2108831
#1 String: Yucaipa owned Dominick's before selling the chain to Safeway in 1998 for $2.5 billion.
#2 String: Yucaipa bought Dominick's in 1995 for $693 million and sold it to Safeway for $1.8 billion in 1998.


Quality: 1
#1 ID: 1330381
#2 ID: 1330521
#1 String: They had published an advertisement on the Internet on June 10, offering the cargo for sale, he added.
#2 String: On June 10, the ship's owners had published an advertisement on the Internet, offering the explosives for sale.


Quality: 0
#1 ID: 3344667
#2 ID: 3344648
#1 String: Around 0335 GMT, 

Huggingface dataset과의 큰 차이는 없어보입니다.

Huggingface dataset가 이중 딕셔너리 내부에 데이터를 리스트 형태로 담았기에, 이와 같은 방식으로 custom dataset을 재구성합니다.

In [9]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# train 데이터를 train(80%)과 validation(20%)으로 분할
train_indices = list(range(len(train_dataset['Quality'])))
train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42, 
                                      stratify=train_dataset['Quality'])

# validation 데이터셋 생성
validation_dataset = {}
for key in train_dataset.keys():
    validation_dataset[key] = [train_dataset[key][i] for i in val_idx]

# train 데이터셋 업데이트 (validation으로 사용된 데이터 제거)
train_dataset_final = {}
for key in train_dataset.keys():
    train_dataset_final[key] = [train_dataset[key][i] for i in train_idx]

# 허깅페이스 Dataset 객체로 변환
train_hf_dataset = Dataset.from_dict(train_dataset_final)
validation_hf_dataset = Dataset.from_dict(validation_dataset)
test_hf_dataset = Dataset.from_dict(test_dataset)

# DatasetDict 생성 (이게 허깅페이스의 표준 방식)
customized_mrpc_dataset = DatasetDict({
    'train': train_hf_dataset,
    'validation': validation_hf_dataset,
    'test': test_hf_dataset
})

# 결과 출력
print("DatasetDict({")
for split_name, split_data in customized_mrpc_dataset.items():
    print(f"    {split_name}: Dataset({{")
    print(f"        features: {list(split_data.features.keys())},")
    print(f"        num_rows: {split_data.num_rows}")
    print("    })")
print("})")

print("\n데이터셋 정보 확인!")
print(f"Train dataset shape: ({customized_mrpc_dataset['train'].num_rows}, {len(customized_mrpc_dataset['train'].features)})")
print(f"Validation dataset shape: ({customized_mrpc_dataset['validation'].num_rows}, {len(customized_mrpc_dataset['validation'].features)})")
print(f"Test dataset shape: ({customized_mrpc_dataset['test'].num_rows}, {len(customized_mrpc_dataset['test'].features)})")

print("Dataset 생성 완료!")
print(f"Train samples: {len(customized_mrpc_dataset['train'])}")
print(f"Validation samples: {len(customized_mrpc_dataset['validation'])}")
print(f"Test samples: {len(customized_mrpc_dataset['test'])}")

# 첫 번째 샘플 확인
print(f"\n첫 번째 train 샘플:")
print(customized_mrpc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 3260
    })
    validation: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 816
    })
    test: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 1725
    })
})

데이터셋 정보 확인!
Train dataset shape: (3260, 5)
Validation dataset shape: (816, 5)
Test dataset shape: (1725, 5)
Dataset 생성 완료!
Train samples: 3260
Validation samples: 816
Test samples: 1725

첫 번째 train 샘플:
{'Quality': 0, '#1 ID': 2121506, '#2 ID': 2121285, '#1 String': 'The festival kicked off yesterday one day after the Competition Commission delivered its final verdict to the Government on the proposed £4.1 billion merger.', '#2 String': 'The Competition Commission delivered its verdict yesterday on the proposed merger of the two big ITV players, Carlton and Granada.'}


이렇게 우리는 같은 MRPC 데이터셋을

- Huggingface datasets에서 불러도 와보고
- 외부에서 데이터를 가져와 커스터마이징도 해 보았습니다!


# (2) Tokenizer와 Model
## Huggingface Auto Classes를 이용하는 방법
---
데이터셋 커스터마이징 작업을 잘 진행했다면 이미 절반 이상 진행한 것이나 마찬가지입니다. NLP 모델링의 핵심을 이루는 Tokenizer와 Model은 framework에서 이미 잘 만들어져 있는 것을 쉽게 가져다 쓸 수 있기 때문입니다.

이번엔 Huggingface에서 Auto Model과 이에 상응하는 tokenizer을 불러오겠습니다.

In [10]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

huggingface_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
huggingface_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Huggingface의 경우 AutoTokenizer, AutoModel기능을 지원합니다.

AutoTokenizer와 AutoModel은 Huggingface에서 지원하는 Auto Class입니다.

Auto class는 from_pretrained 메소드를 이용해 pretrained model의 경로 혹은 이름만 안다면 자동으로 생성하는 방법입니다.

즉 bert를 사용할때 BertTokenizer, RoBERTa를 사용할때 RobertaTokenizer를 사용하게 되는데 AutoTokenizer를 이용하면 자동으로 BERT모델은 BERT로 RoBERTa모델은 RoBERTa로 바꿔줍니다.

model도 마찬가지입니다. 다만 model의 경우 AutoModel을 그대로 사용하기보다 특정 task를 지정하는 방식인 AutoModelForSequenceClassification을 사용하는걸 권장드립니다.

Auto class는 다양한 모델에 자동으로 맞출 수 있기 때문에 특정 task와 dataset이 주어져있는 경우 모델을 다양하게 넣어 실험할 수 있습니다.

그렇기에 Auto class를 유용하게 활용하는 것을 추천합니다.

Tokenizer와 Model을 만들었다면 이제 토크나이징하는 방법을 알아보도록 하겠습니다.

토크나이징은 transform이라는 함수를 만들고 이전에 만들어두었던 Tokenizer를 사용하는데 이때 dataset의 형태를 확인하고 바꿀 대상을 지정해야 합니다.

mrpc의 경우 sentence1, sentence2가 토크나이징할 대상이므로 data[’sentence1’], data[’sentence2’]로 인덱싱해서 지정합니다.

truncation은 특정 문장이 길어 모델을 다루기 힘들어 질 수 있으므로 짧게 자르는 것을 의미합니다.

return_token_type_ids는 문장이 한개이상일 때 나뉘는걸 보여줍니다. (해당 내용은 task에 필요없으므로 제거합니다)

In [11]:
def transform(data):
    return huggingface_tokenizer(
        data['sentence1'],
        data['sentence2'],
        truncation = True,
        padding = 'max_length',
        return_token_type_ids = False,
        )

데이터셋을 한번에 토크나이징할때 자주 사용하는 기법은 map입니다.

map을 사용하게 되면 Data dictionary에 있는 모든 데이터들이 빠르게 적용시킬 수 있습니다.

우리는 map을 사용해 토크나이징을 진행하기 때문에 batch를 적용해야 되므로 batched=True로 주어야 합니다.

In [12]:
hf_dataset = huggingface_mrpc_dataset.map(transform, batched=True)

# train & validation & test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [13]:
huggingface_mrpc_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

이전 스텝에서 tfds의 MRPC로 만든 커스텀 데이터셋의 경우 그대로 적용할 경우 오류가 발생합니다. 어떤 오류가 발생하는지 확인해볼까요?

In [16]:
# Q. tf_train_dataset에 transform 함수를 매핑해봅시다. 어떤 오류가 발생하나요?
# tf_train_dataset_error = tf_train_dataset.map(transform)

tf_train_dataset_error = train_df.map(transform)

TypeError: 'int' object is not subscriptable

보아하니 데이터 타입이 맞지 않아 발생하는 오류로 보입니다. 이 경우 기존 transform 함수가 sentence를 불러올 때 디코딩을 해주면 해결됩니다.

새로운 함수 transform_custom 를 아래와 같이 정의하고 커스텀 데이터셋에 적용해보겠습니다.

In [17]:
# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# validation set이 없는 경우, train set에서 일부를 분할
# 또는 test set을 validation으로 사용
val_dataset = test_dataset.copy()  # 예시로 test를 val로 복사

# Huggingface Dataset 생성
train_dataset = Dataset.from_dict(train_dataset)
val_dataset = Dataset.from_dict(val_dataset) 
test_dataset = Dataset.from_dict(test_dataset)

print("Dataset 생성 완료!")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# 커스텀 데이터용 transform 함수 정의
def transform_custom(batch):
    # 커스텀 파일 데이터는 이미 문자열이므로 decode 불필요
    sentence1 = batch['#1 String']
    sentence2 = batch['#2 String']
    return huggingface_tokenizer(
        sentence1,
        sentence2,
        truncation=True,
        padding='max_length',
        return_token_type_ids=False,
    )

# 토큰화 및 패딩을 적용
train_dataset = train_dataset.map(transform_custom, batched=True)
val_dataset = val_dataset.map(transform_custom, batched=True)
test_dataset = test_dataset.map(transform_custom, batched=True)

Dataset 생성 완료!
Train samples: 4076
Validation samples: 1725
Test samples: 1725


Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

어떤가요? 정상적으로 토크나이징이 되는 것을 확인할 수 있습니다. 모델과 토큰화된 데이터가 준비되었으니 이제 남은 것은 학습과 추론 뿐입니다.

## (3) Train/Evaluation과 Test
### Trainer를 활용한 학습
---
이제 다 왔습니다! Huggingface의 Trainer을 활용해 학습을 진행해보도록 하겠습니다.

Trainer를 사용하기 위해서는 TrainingArguments를 통해 학습 관련 설정을 미리 지정해야 합니다.

In [19]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments

output_dir = 'transformers'

training_arguments = TrainingArguments(
    output_dir,                                         # output이 저장될 경로
    eval_strategy="epoch",           #evaluation하는 빈도
    learning_rate = 2e-5,                         #learning_rate
    per_device_train_batch_size = 8,   # 각 device 당 batch size
    per_device_eval_batch_size = 8,    # evaluation 시에 batch size
    num_train_epochs = 3,                     # train 시킬 총 epochs
    weight_decay = 0.01,                        # weight decay
)

아래에서 생성하게 될 Trainer의 인자로 넘겨주어야 할 것 중에 compute_metrics 메소드가 있습니다.

이것은 task가 classification인지 regression인지에 따라 모델의 출력 형태가 달라지므로 task별로 적합한 출력 형식을 고려해 모델의 성능을 계산하는 방법을 미리 지정해 두는 것입니다.

MRPC 데이터셋은 binary classification에 해당하겠죠?

In [20]:
!pip install evaluate

In [21]:
from evaluate import load
metric = load('glue', 'mrpc')

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references = labels)

Trainer에 model, arguments, train_dataset, eval_dataset, compute_metrics를 넣고 train을 진행합니다.

In [22]:
trainer = Trainer(
    model=huggingface_model,           # 학습시킬 model
    args=training_arguments,           # TrainingArguments을 통해 설정한 arguments
    train_dataset=hf_train_dataset,    # training dataset
    eval_dataset=hf_val_dataset,       # evaluation dataset
    compute_metrics=compute_metrics,
)
trainer.train()
print("슝~")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.367996,0.840686,0.886165
2,0.510759,0.428827,0.840686,0.890017
3,0.323393,0.529554,0.867647,0.907850


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

슝~


마지막으로 test 데이터셋으로 평가를 합니다.

In [24]:
# trainer.evaluate(hf_test_dataset)

# Trainer 정의 후 실행 전 아래 한 줄을 추가하여 콜백을 강제로 제거
from transformers.utils.notebook import NotebookProgressCallback

trainer.remove_callback(NotebookProgressCallback)

trainer.evaluate(hf_test_dataset)

RuntimeError: on_train_begin must be called before on_evaluate

### 커스텀 데이터셋으로 학습
방금 학습시킨 모델은 Huggingface dataset으로 학습시켰는데요, 앞에서 우리가 만든 커스텀 데이터셋 (tf_train_dataset, tf_val_dataset, tf_test_dataset)으로도 학습을 진행해보고, 결과를 비교해봅시다.

In [25]:
#메모리를 비워줍니다.
del huggingface_model

In [27]:
# Q. 커스텀 데이터셋으로 학습시켜봅시다.
huggingface_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)

trainer_custom = Trainer(
    model=huggingface_model,
    args=training_arguments,
    train_dataset=train_dataset,  # tf_train_dataset,
    eval_dataset=val_dataset,     # tf_val_dataset,
    compute_metrics=compute_metrics,
)
trainer_custom.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [28]:
# Validation 데이터셋을 이용해 평가해봅니다.
eval_results = trainer_custom.evaluate()

print(eval_results)

Epoch,Training Loss,Validation Loss
0,No log,No log


{}


이렇게 데이터셋 구성부터 전처리와 학습까지, Huggingface 모델의 flow를 압축하여 알아보았습니다. 이제 프로젝트로 이동하여 조금 더 익숙해져보는 시간을 가져봅시다! 수고하셨습니다 👏👏👏

# 4. 프로젝트 : 커스텀 프로젝트 직접 만들기
실습 코드에서 수행해 본 내용을 토대로, 이번에는 한국어 데이터셋에 도전해보겠습니다.

앞서 본 GLUE benchmark의 한국어 버전 [KLUE benchmark](https://klue-benchmark.com/)를 들어보신 적 있나요?

GLUE와 마찬가지로 한국어 자연어처리에 대한 이해도를 높이기 위해 만들어진 데이터셋 benchmark입니다. 총 8가지의 데이터셋이 있습니다. 다만 이번 시간에 진행할 프로젝트는 KLUE의 dataset을 활용하는 것이 아닌, model(klue/ber-base)를 활용하여 NSMC(Naver Sentiment Movie Corpus) task를 도전해보겠습니다.

모델과 데이터에 관한 정보는 링크를 참조해주세요.

- [KLUE/Bert-base](https://huggingface.co/klue/bert-base)
- [NSMC](https://github.com/e9t/nsmc)

준비가 되셨다면 아래와 같은 순서로 진행해주세요.

라이브러리 버전을 확인해 봅니다.
사용할 라이브러리 버전을 둘러봅시다.

<details>
<summary>model(klue/ber-base)</summary>

klue/bert-base는 한국어에 특화된 BERT base 모델로, KLUE(Korean Language Understanding Evaluation) 벤치마크를 위해 Naver와 KAIST가 개발한 사전 학습된 언어 모델입니다.

- 주요 특징
    - 아키텍처: 표준 BERT-base와 동일한 12층 트랜스포머 인코더 구조(768 hidden size, 12 attention heads)를 가지며, 한국어 말뭉치(모두의 말뭉치, CC-100-Kor, 나무위키, 뉴스 등)로 사전 학습되었습니다.

    - 성능: KLUE 리더보드에서 한국어 NLU 태스크(의미적 유사성, NER, RE 등)에서 기존 KoBERT나 다국어 모델을 능가하는 baseline 성능을 보입니다.

    - 용도: 한국어 분류, NER, paraphrase detection(MRPC 같은 태스크) fine-tuning에 최적화되어 있으며, 이전에 논의한 MRPC 데이터셋과 잘 맞습니다.

- 사용 예시
Hugging Face에서 from_pretrained("klue/bert-base")로 쉽게 로드 가능하며, KLUE 데이터셋과 함께 한국어 NLP 프로젝트에 자주 쓰입니다.
</details>

<details>
<summary>NSMC(Naver Sentiment Movie Corpus</summary>

NSMC(Naver Sentiment Movie Corpus)는 네이버 영화 리뷰를 기반으로 한 한국어 감성 분석 데이터셋입니다. 영화 리뷰 텍스트에 대해 긍정(1) 또는 부정(0) 라벨을 붙인 이진 분류용 데이터로, 총 20만 개 샘플(훈련 15만, 테스트 5만)을 포함합니다.

- 주요 특징
    - 데이터 출처: 네이버 영화 리뷰(140자 미만)에서 긍정(9-10점), 부정(0-4점)으로 라벨링.

    - 용도: 한국어 감성 분석 모델 fine-tuning에 표준적으로 사용되며, klue/bert-base 같은 모델과 함께 자주 활용됩니다.

    - 로드 방법: GitHub(e9t/nsmc)에서 다운로드하거나 Korpora 라이브러리로 쉽게 불러올 수 있습니다.

이 데이터셋은 MRPC나 KLUE 태스크와 함께 한국어 NLP 실습에 필수적입니다.
</details>

In [29]:
import tensorflow
import numpy
import transformers
import datasets

print(tensorflow.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(datasets.__version__)

ModuleNotFoundError: No module named 'tensorflow'

### STEP 1. NSMC 데이터 분석 및 Huggingface dataset 구성
- 데이터셋은 깃허브에서 다운받거나, Huggingface datasets에서 가져올 수 있습니다. 앞에서 배운 방법들을 활용해봅시다!
### STEP 2. klue/bert-base model 및 tokenizer 불러오기
### STEP 3. 위에서 불러온 tokenizer으로 데이터셋을 전처리하고, model 학습 진행해 보기
### STEP 4. Fine-tuning을 통하여 모델 성능(accuarcy) 향상시키기
- 데이터 전처리, TrainingArguments 등을 조정하여 모델의 정확도를 90% 이상으로 끌어올려봅시다.
### STEP 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와의 비교
- 아래 링크를 바탕으로 bucketing과 dynamic padding이 무엇인지 알아보고, 이들을 적용하여 model을 학습시킵니다.
    - [Data Collator](https://huggingface.co/docs/transformers/v4.30.0/en/main_classes/data_collator)
    - [Trainer.TrainingArguments 의 group_by_length](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)


- STEP 4에 학습한 결과와 bucketing을 적용하여 학습시킨 결과를 비교해보고, 모델 성능 향상과 훈련 시간 두 가지 측면에서 각각 어떤 이점이 있는지 비교해봅시다.

<details>
<summary>bucketing</summary>

Bucketing은 NLP 모델 학습 시 입력 시퀀스(토큰 길이)를 비슷한 길이 그룹(버킷)으로 나누어 패딩을 최소화하는 데이터 전처리 기법입니다.

- 주요 특징
    - 목적: 고정 길이 패딩 대신 동적 패딩으로 메모리 효율과 학습 속도를 높임. 예를 들어, 길이 10~20, 20~30 버킷으로 나눔.

    - 적용 예: KLUE/NSMC/MRPC 같은 한국어 데이터셋에서 klue/bert-base fine-tuning 시 DataLoader에서 자주 사용되며, max_pad_len으로 버킷 크기 조절.

    - 장점: 긴 시퀀스 패딩 낭비 줄여 배치 처리 최적화; PyTorch BucketSampler나 Hugging Face Trainer에서 지원.

이전 논의한 데이터셋(NSMC, MRPC) 처리 시 토큰 길이 다양성을 bucketing으로 해결할 수 있습니다.
</details>

<details>
<summary>dynamic padding </summary>

Dynamic padding은 NLP 모델 학습 시 각 배치마다 해당 배치 내 최대 시퀀스 길이에만 패딩을 적용하는 기법입니다.

- 주요 특징
    - 기존 패딩 vs Dynamic: 고정 길이(예: 512) 패딩 대신 배치별 동적 결정으로 불필요한 패딩 최소화.

    - Bucketing과 연계: 이전에 설명한 bucketing(길이 그룹화)과 함께 사용되며, klue/bert-base + NSMC/MRPC fine-tuning에서 메모리 효율 극대화.

    - 구현: PyTorch pad_sequence()나 Hugging Face DataCollatorWithPadding에서 자동 지원; attention mask로 패딩 무시.

이 기법으로 NSMC 리뷰(짧음)와 MRPC 문장쌍(길이 다양)을 효율적으로 배치 처리할 수 있습니다.
</details>